# F — Full experiment (GPU)

One run: generate tutor answers, verify them against the curriculum graph, and score the
two baselines. Replaces notebooks C, D and E, which stay in the repo as the development
history that produced the fixes below.

### What the first run exposed, and what changed

**30% of generated answers were unusable.** 37% of the 0.5B answers echoed the prompt's
rule list back verbatim, and 35% of the 1.5B answers were decoding loops. Fixed by adding
`repetition_penalty`, moving the rules into the system message so there is no list in the
user turn to copy, and discarding answers that still degenerate.

**Lexical matching had a hard ceiling.** 56% of claims that linked to a curriculum entity
had *zero* word overlap with any fact about it — not a threshold problem:

```
claim    : হৃৎপিণ্ড গোলাকার একটি পেশি দ্বারা গঠিত
evidence : হৃৎপেশি দিয়ে গঠিত          ← same statement, 0.33 lexical overlap
```

Bangla compounding makes this systematically worse than in English. Matching is now
lexical **and** embedding-based, taking the stronger signal of the two.

**Scores are saved, not just verdicts,** so the decision threshold can be swept after the
manual labelling rather than being guessed now.

**Settings:** GPU `NvidiaTeslaT4`, Internet on, dataset `bangla-biology-kg`,
models `qwen-lm/qwen2.5` at 0.5b / 1.5b / 3b / 7b-instruct.

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate sentence-transformers

import importlib.util
for pkg in ("bitsandbytes", "sentence_transformers"):
    assert importlib.util.find_spec(pkg), f"{pkg} missing — restart the session"
print("deps ready")

In [ ]:
import json, glob, os, gc, re, time, collections
from pathlib import Path
import pandas as pd
import numpy as np

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- config -------------------------------------------------------------
TUTORS   = ["0.5b-instruct", "1.5b-instruct", "3b-instruct", "7b-instruct"]
JUDGE    = "7b-instruct"        # reuses the tutor model already in memory
# LaBSE, not MiniLM. On 10 hand-built Bangla pairs MiniLM scored unrelated pairs
# at 0.69-0.88 and true matches at 0.65-0.97 - overlapping, so no threshold works.
# LaBSE puts negatives at 0.12-0.58 against positives at 0.69-0.89.
EMBEDDER = "sentence-transformers/LaBSE"

GEN_MAX_NEW  = 256
GEN_BATCH    = 8
REP_PENALTY  = 1.15      # 0 of 680 first-run answers used one; 20% were loops
JUDGE_MAX_NEW = 96
JUDGE_BATCH  = 8
TOP_K        = 3         # passages shown to the retrieval baseline

SIM_SUPPORT  = 0.65      # midway between LaBSE's observed pos/neg bands;
                         # provisional - the labelling round sets it properly
MIN_CLAIM_WORDS = 3
LIMIT = None             # questions; None = all 170
# -------------------------------------------------------------------------

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("eval")
WORK.mkdir(parents=True, exist_ok=True)
ANSWERS_JL = WORK / "tutor_answers.jsonl"
JUDGE_JL   = WORK / "baseline_judgements.jsonl"


def find(name, *fallbacks):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for f in fallbacks:
        hits += glob.glob(f)
    if not hits:
        raise SystemExit(f"{name} not found — attach the bangla-biology-kg dataset")
    return hits[0]


Q = pd.read_csv(find("biology_eval_questions.csv", "eval/*.csv", "../eval/*.csv"))
T = pd.read_csv(find("biology_all_triples.csv", "kg/triples/*.csv", "../kg/triples/*.csv"))
CORPUS = pd.read_parquet(find("biology_clean.parquet", "eval/*.parquet", "../eval/*.parquet"))
T["subject"] = T.subject.astype(str).str.strip()
T["object"] = T.object.astype(str).str.strip()
if LIMIT:
    Q = Q.head(LIMIT)

print(f"{len(Q)} questions | {len(T)} triples | {len(CORPUS)} passages")
print(f"{len(Q) * len(TUTORS)} answers to generate")

## 1 — Tutor answers

The rules live in the system message. In the first run they were a numbered list in the
user turn, and the 0.5B model simply continued the list instead of answering.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

TUTOR_SYSTEM = (
    "তুমি বাংলাদেশের নবম-দশম শ্রেণির জীববিজ্ঞান বিষয়ের একজন শিক্ষক। "
    "শিক্ষার্থীর প্রশ্নের উত্তর সহজ বাংলায়, তিন থেকে চার বাক্যে দাও। "
    "পাঠ্যবইয়ের তথ্য অনুযায়ী উত্তর দেবে। কোনো নিয়ম বা নির্দেশনা পুনরাবৃত্তি করবে না।")


def load_model(size):
    cfgs = glob.glob(f"/kaggle/input/**/{size}/**/config.json", recursive=True)
    if not cfgs:
        raise SystemExit(f"{size} not attached")
    path = str(Path(cfgs[0]).parent)
    assert f"/{size}/" in path + "/", f"resolved wrong model: {path}"
    tok = AutoTokenizer.from_pretrained(path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    m = AutoModelForCausalLM.from_pretrained(
        path,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True),
        device_map="auto").eval()
    return tok, m


@torch.inference_mode()
def gen(tok, model, prompts, max_new):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                         repetition_penalty=REP_PENALTY,
                         pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
            for o in out]


def safe_gen(tok, model, prompts, max_new):
    try:
        return gen(tok, model, prompts, max_new)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("\n  OOM — one at a time")
        return [gen(tok, model, [p], max_new)[0] for p in prompts]


done = set()
if ANSWERS_JL.exists():
    with open(ANSWERS_JL, encoding="utf-8") as f:
        done = {(r["model"], r["qid"]) for r in map(json.loads, filter(str.strip, f))}
    print(f"resuming — {len(done)} answers present")

judge_tok = judge_model = None
t_all = time.time()

with open(ANSWERS_JL, "a", encoding="utf-8") as sink:
    for size in TUTORS:
        todo = Q[[(size, q) not in done for q in Q.qid]].reset_index(drop=True)
        if not len(todo):
            print(f"{size}: complete")
            if size == JUDGE and judge_model is None:
                judge_tok, judge_model = load_model(size)
            continue
        tok, model = load_model(size)
        print(f"\n{size}: {len(todo)} answers")
        t0 = time.time()
        for i in range(0, len(todo), GEN_BATCH):
            part = todo.iloc[i:i + GEN_BATCH]
            prompts = [tok.apply_chat_template(
                [{"role": "system", "content": TUTOR_SYSTEM},
                 {"role": "user", "content": q}],
                tokenize=False, add_generation_prompt=True) for q in part.question_text]
            for row, ans in zip(part.itertuples(), safe_gen(tok, model, prompts, GEN_MAX_NEW)):
                sink.write(json.dumps({
                    "model": size, "qid": row.qid, "chapter_no": int(row.chapter_no),
                    "question": row.question_text, "answer": ans}, ensure_ascii=False) + "\n")
            sink.flush()
            n = min(i + GEN_BATCH, len(todo))
            print(f"  {n}/{len(todo)}  {time.time()-t0:.0f}s", end="\r")
        print(f"\n  done in {time.time()-t0:.0f}s")
        # Keep the judge in memory instead of paying to load it twice.
        if size == JUDGE:
            judge_tok, judge_model = tok, model
        else:
            del model, tok
            gc.collect(); torch.cuda.empty_cache()

A = pd.DataFrame([json.loads(l) for l in open(ANSWERS_JL, encoding="utf-8") if l.strip()])
A = A.drop_duplicates(subset=["model", "qid"])
print(f"\n{len(A)} answers in {time.time()-t_all:.0f}s")

## 2 — Discard degenerate answers

An answer that loops a phrase or recites the instructions contains no assertion to verify.
Counting it as "not in curriculum" would inflate that class and flatter the method, so
these are removed and **reported** — the rate is itself a finding about small models.

In [ ]:
ECHO = re.compile(r"উত্তর বাংলায় লিখবে|বাক্যের মধ্যে|পাঠ্যবইয়ের তথ্য অনুযায়ী"
                  r"|নিচের প্রশ্নটির উত্তর দাও|নিয়ম:|পুনরাবৃত্তি করবে না")


def is_loop(s, min_rep=3):
    w = str(s).split()
    if len(w) < 12:
        return False
    for n in (3, 4, 5):
        g = [" ".join(w[i:i + n]) for i in range(len(w) - n + 1)]
        if g and pd.Series(g).value_counts().iloc[0] >= min_rep:
            return True
    return False


A["echo"] = A.answer.astype(str).str.contains(ECHO)
A["loop"] = A.answer.map(is_loop)
A["usable"] = ~(A.echo | A.loop) & (A.answer.astype(str).str.len() > 15)

print("answer quality by model (%):")
print((A.groupby("model")[["echo", "loop", "usable"]].mean() * 100).round(1).to_string())
print(f"\nusable overall: {A.usable.sum()}/{len(A)} ({A.usable.mean()*100:.1f}%)")
A.to_csv(WORK / "tutor_answers.csv", index=False)

## 3 — Claims

In [ ]:
WORD = re.compile(r"[ঀ-৿]+|[A-Za-z]+|[0-9০-৯]+(?:[.,][0-9০-৯]+)*")
NUMBER = re.compile(r"[0-9০-৯]+(?:[.,][0-9০-৯]+)*")
SENT = re.compile(r"(?<=[।?!])\s+|\n+|(?:^|\s)[-*•]\s+")
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
SUFFIXES = ["গুলোর", "গুলোকে", "গুলো", "টিকে", "গুলি", "দের", "টির", "টি",
            "য়ের", "এর", "কে", "ের", "রা", "র"]
STOP = {"এবং", "বা", "এই", "সেই", "এটি", "এটা", "যে", "যা", "তা", "হয়", "হলো",
        "করে", "থেকে", "জন্য", "সাথে", "মধ্যে", "সব", "অনেক", "কিছু", "করা",
        "হয়ে", "একটি", "একটা", "নয়", "না", "তাই", "কিন্তু", "আর", "ও"}


def lemma(w):
    for s in SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 3:
            return w[: -len(s)]
    return w


def toks(s):
    return [lemma(w) for w in WORD.findall(str(s))]


def content(s):
    return {w for w in toks(s) if w not in STOP and len(w) >= 3}


def numbers(s):
    return {n.translate(BN_DIGITS).rstrip(".,") for n in NUMBER.findall(str(s))}


rows = []
for r in A[A.usable].itertuples():
    for j, s in enumerate(SENT.split(str(r.answer))):
        s = s.strip(" ।\t")
        if len(toks(s)) >= MIN_CLAIM_WORDS:
            rows.append({"model": r.model, "qid": r.qid, "claim_no": j,
                         "chapter_no": r.chapter_no, "question": r.question, "claim": s})
C = pd.DataFrame(rows)
print(f"{len(C)} claims from {A.usable.sum()} usable answers")
print(C.groupby("model").size().rename("claims").to_string())

## 4 — Graph verification

Entity linking is whole-token. Each claim is then scored against every fact about its
entities two ways — content-word overlap and embedding cosine — and the stronger signal
wins, because `হৃৎপেশি` versus `হৃৎপিণ্ড ... পেশি` is a compounding difference, not a
difference in meaning.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDER, device="cuda")
print("embedder loaded:", EMBEDDER)

GENERIC = {"মাধ্যম", "ধরন", "জিনিস", "সময়", "স্থান", "গুরুত্ব", "নাম",
           "ব্যাপার", "কারণ", "ফলে", "দিক", "অবস্থা", "অংশ"}

by_len = collections.defaultdict(dict)
for e in T.subject.unique():
    tk = tuple(toks(e))
    if tk and len("".join(tk)) >= 4 and e not in GENERIC:
        by_len[len(tk)][tk] = e
MAXN = max(by_len)

FACTS = collections.defaultdict(list)
for i, r in enumerate(T.itertuples()):
    FACTS[r.subject].append(i)


def link(text):
    tk, found, covered = toks(text), [], set()
    for n in range(MAXN, 0, -1):
        tab = by_len.get(n)
        if not tab:
            continue
        for i in range(len(tk) - n + 1):
            if any(j in covered for j in range(i, i + n)):
                continue
            hit = tab.get(tuple(tk[i:i + n]))
            if hit:
                found.append(hit)
                covered.update(range(i, i + n))
    return found


C["entities"] = C.claim.map(link)
print(f"claims linked to an entity: {(C.entities.map(len) > 0).mean()*100:.0f}%")

# Encode facts as "subject relation object" so the entity is part of the vector.
fact_text = (T.subject + " " + T.relation + " " + T.object).tolist()
F_emb = embedder.encode(fact_text, batch_size=128, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=False)
C_emb = embedder.encode(C.claim.astype(str).tolist(), batch_size=128,
                        convert_to_numpy=True, normalize_embeddings=True,
                        show_progress_bar=False)
print(f"encoded {len(F_emb)} facts and {len(C_emb)} claims")

In [ ]:
out = []
for ci, r in enumerate(C.itertuples()):
    cw, cn = content(r.claim), numbers(r.claim)
    cand = sorted({i for e in r.entities for i in FACTS.get(e, [])})
    best = {"verdict": "not_in_curriculum", "lex": 0.0, "sim": 0.0, "score": 0.0,
            "entity": r.entities[0] if r.entities else "", "relation": "",
            "evidence": "", "evidence_id": ""}
    if cand:
        sims = F_emb[cand] @ C_emb[ci]
        contradiction = None
        for k, fi in enumerate(cand):
            f = T.iloc[fi]
            fw = content(f.object)
            lex = len(fw & cw) / len(fw) if fw else 0.0
            sim = float(sims[k])
            # A stated quantity that disagrees with the stored one is the clearest
            # contradiction a curriculum graph can detect; it outranks any text match.
            fn = numbers(f.object)
            if f.relation == "পরিমাণ" and cn and fn and not (cn & fn):
                contradiction = {"verdict": "contradicted", "lex": lex, "sim": sim,
                                 "score": 1.0, "entity": f.subject, "relation": f.relation,
                                 "evidence": f.object, "evidence_id": f.triple_id}
                break
            score = max(lex, sim)
            if score > best["score"]:
                best = {"verdict": "supported" if score >= SIM_SUPPORT else "not_in_curriculum",
                        "lex": round(lex, 3), "sim": round(sim, 3), "score": round(score, 3),
                        "entity": f.subject, "relation": f.relation,
                        "evidence": f.object, "evidence_id": f.triple_id}
        if contradiction:
            best = contradiction
    out.append(best)

C = pd.concat([C.drop(columns=["entities"]), pd.DataFrame(out)], axis=1)
C["system"] = "D_graph"
C.to_csv(WORK / "claim_verdicts.csv", index=False)

print(C.verdict.value_counts().to_string())
print("\nby model (%):")
print(pd.crosstab(C.model, C.verdict, normalize="index").mul(100).round(1).to_string())
print("\nthreshold sweep on the max(lexical, embedding) score:")
for th in [0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8]:
    print(f"  >= {th:.2f} -> supported {(C.score >= th).mean()*100:5.1f}%")

## 5 — Baselines

Same claims, same label space. E1 judges from parametric knowledge — the BenHalluEval
analogue. E2 additionally sees the top-3 TF-IDF passages from the same corpus the graph
was built from, so it differs from the graph only in representation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vec = TfidfVectorizer(analyzer="word", token_pattern=r"[ঀ-৿]+|[A-Za-z]+",
                      min_df=2, sublinear_tf=True)
M = vec.fit_transform(CORPUS.text.astype(str))
sims = linear_kernel(vec.transform(C.claim.astype(str)), M)
top = sims.argsort(axis=1)[:, ::-1][:, :TOP_K]
PASSAGES = [[str(CORPUS.text.iloc[j])[:600] for j in row] for row in top]
print(f"retrieval ready: {M.shape[0]} passages x {M.shape[1]} terms")

LABELS = {"সমর্থিত": "supported", "বিরোধী": "contradicted",
          "পাঠ্যক্রমে_নেই": "not_in_curriculum"}
JUDGE_SYSTEM = ("তুমি নবম-দশম শ্রেণির জীববিজ্ঞান পাঠ্যবইয়ের একজন বিশেষজ্ঞ। "
                "তুমি শুধুমাত্র বৈধ JSON উত্তর দাও।")
RULES = """নিচের দাবিটি নবম-দশম শ্রেণির জীববিজ্ঞান পাঠ্যক্রম অনুযায়ী যাচাই করো।

- "সমর্থিত": দাবিটি পাঠ্যক্রমের তথ্যের সাথে মিলে যায়।
- "বিরোধী": দাবিটি পাঠ্যক্রমের তথ্যের বিপরীত।
- "পাঠ্যক্রমে_নেই": দাবিটি সত্য হতে পারে, কিন্তু এই শ্রেণির পাঠ্যক্রমে নেই।

শুধু এই ফরম্যাটে JSON দাও:
{{"রায়": "...", "কারণ": "এক বাক্যে"}}"""


def judge_prompt(tok, claim, passages=None):
    user = RULES + ("" if passages is None else
                    "\n\nপাঠ্যবই থেকে প্রাসঙ্গিক অংশ:\n" +
                    "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages)))
    user += f"\n\nদাবি: {claim}"
    return tok.apply_chat_template(
        [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)


def parse(text):
    t = re.sub(r"^```(?:json)?|```$", "", str(text).strip(), flags=re.M)
    m = re.search(r"\{.*?\}", t, re.S)
    if m:
        try:
            lab = str(json.loads(m.group()).get("রায়", "")).strip().strip('"')
            if lab in LABELS:
                return LABELS[lab]
        except json.JSONDecodeError:
            pass
    for bn, en in LABELS.items():
        if bn in t:
            return en
    return None

In [ ]:
if judge_model is None:
    judge_tok, judge_model = load_model(JUDGE)

seen = set()
if JUDGE_JL.exists():
    with open(JUDGE_JL, encoding="utf-8") as f:
        seen = {(r["system"], r["model"], r["qid"], r["claim_no"])
                for r in map(json.loads, filter(str.strip, f))}
    print(f"resuming — {len(seen)} judgements present")

t_all = time.time()
with open(JUDGE_JL, "a", encoding="utf-8") as sink:
    for system, use_p in [("E1_closed_book", False), ("E2_retrieval", True)]:
        idx = [i for i, r in enumerate(C.itertuples())
               if (system, r.model, r.qid, r.claim_no) not in seen]
        if not idx:
            print(f"{system}: complete")
            continue
        print(f"\n{system}: {len(idx)} claims")
        t0 = time.time()
        for s in range(0, len(idx), JUDGE_BATCH):
            chunk = idx[s:s + JUDGE_BATCH]
            prompts = [judge_prompt(judge_tok, C.claim.iloc[i],
                                    PASSAGES[i] if use_p else None) for i in chunk]
            raws = safe_gen(judge_tok, judge_model, prompts, JUDGE_MAX_NEW)
            for i, raw in zip(chunk, raws):
                r = C.iloc[i]
                sink.write(json.dumps({
                    "system": system, "model": r.model, "qid": r.qid,
                    "claim_no": int(r.claim_no), "claim": r.claim,
                    "verdict": parse(raw), "raw": raw[:250]}, ensure_ascii=False) + "\n")
            sink.flush()
            n = min(s + JUDGE_BATCH, len(idx))
            print(f"  {n}/{len(idx)}  {time.time()-t0:.0f}s", end="\r")
        print(f"\n  done in {time.time()-t0:.0f}s")
print(f"\nbaselines done in {time.time()-t_all:.0f}s")

## 6 — Compare, and emit the labelling sample

In [ ]:
B = pd.DataFrame([json.loads(l) for l in open(JUDGE_JL, encoding="utf-8") if l.strip()])
B = B.drop_duplicates(subset=["system", "model", "qid", "claim_no"])
print(f"{len(B)} judgements, {B.verdict.isna().sum()} unparseable")

cols = ["model", "qid", "claim_no", "claim", "verdict", "system"]
ALL = pd.concat([C[cols], B[cols]], ignore_index=True)
ALL.to_csv(WORK / "all_system_verdicts.csv", index=False)

print("\nverdict distribution by system (%):")
print(pd.crosstab(ALL.system, ALL.verdict, normalize="index").mul(100).round(1).to_string())

wide = ALL.pivot_table(index=["model", "qid", "claim_no"], columns="system",
                       values="verdict", aggfunc="first").dropna()
print(f"\nclaims all three systems judged: {len(wide)}")
for a, b in [("D_graph", "E1_closed_book"), ("D_graph", "E2_retrieval"),
             ("E1_closed_book", "E2_retrieval")]:
    if a in wide.columns and b in wide.columns:
        print(f"  {a} vs {b}: agree {(wide[a] == wide[b]).mean()*100:.1f}%")
print("\nAgreement is not accuracy — which system is right needs the labels below.")

In [ ]:
# Sample where the systems DISAGREE plus a random slice, so labelling effort goes where
# it separates the systems rather than confirming cases they already agree on.
dis = wide[(wide.get("D_graph") != wide.get("E1_closed_book"))
           | (wide.get("D_graph") != wide.get("E2_retrieval"))].reset_index()
agree = wide.drop(index=dis.set_index(["model", "qid", "claim_no"]).index,
                  errors="ignore").reset_index()

take_dis = dis.sample(min(150, len(dis)), random_state=13)
take_agr = agree.sample(min(50, len(agree)), random_state=13)
S = pd.concat([take_dis, take_agr]).merge(
    C[["model", "qid", "claim_no", "claim", "question", "entity", "evidence", "score"]],
    on=["model", "qid", "claim_no"], how="left").sample(frac=1, random_state=13)

S = S.reset_index(drop=True)
S.insert(0, "triple_id", [f"V{i:04d}" for i in range(1, len(S) + 1)])
S["subject"] = S.claim
S["relation"] = S.get("D_graph", pd.Series([""] * len(S)))
S["object"] = S.evidence.fillna("")
S["source_text"] = ("প্রশ্ন: " + S.question.fillna("").astype(str)
                    + "   |   entity: " + S.entity.fillna("").astype(str)
                    + "   |   score: " + S.score.fillna(0).astype(str))

keep = ["triple_id", "model", "subject", "relation", "object", "source_text"]
S[keep].to_csv(WORK / "verdict_review_sample.csv", index=False)
print(f"{len(S)} verdicts to label -> verdict_review_sample.csv "
      f"({len(take_dis)} where systems disagree, {len(take_agr)} where they agree)")
print("\nIn annotate.html: 1 = the graph verdict is right, 3 = wrong, 2 = right call,")
print("wrong evidence. Labelling the disagreements is what separates the systems.")